# 🧩 ARC Logic Decoder

This tool converts complex ARC-AGI grids into a **structured list of building blocks**. Instead of reading long descriptions, it helps you identify the exact logic needed to build the smallest, most efficient AI models for the competition.

### 📋 Table of Contents
1. [**Task Decoder**](#task-decoder) – Labels each task with its specific logic and transformation types.
2. [**Output Preview**](#output-preview) – Shows the task list, transformation counts, difficulty scores, and grid size changes.
3. [**Logic Visualizer**](#logic-visualizer) – Compares the produced logic results directly with the task images.

---

### 🚀 Why this is helpful
* **Pick the Best Tools:** Identify which transformations (like "Mirror" or "Shift") are used. This helps you categorize tasks and see where your code succeeds or struggles.
* **Win Faster:** Use the **Complexity (1-10)** score for a quick evaluation. It helps you find easy tasks and understand why the model might struggle with harder ones.
* **Grid Changes:** Instantly track if a task changes dimensions, giving your model the right information to handle resizing.

### 🔗 Dataset Link
* 📊 **[Get the Full Logic Dataset](https://www.kaggle.com/datasets/karnakbaevarthur/neurogolf-2026-task-transformation-library)** – A single file containing all logic types, grid flags, and difficulty levels for 400 tasks.

---


<a id="task-decoder"></a>
## 🧠 Task Decoder

In [ ]:
import json, os, glob, time, csv
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# --- Configuration ---
START_TASK, END_TASK = 1, 1
BATCH_SIZE = 1  
JSON_OUTPUT = '/kaggle/working/arc_primitives.json'
CSV_OUTPUT = '/kaggle/working/arc_primitives.csv'

# --- API Setup ---
user_secrets = UserSecretsClient()
client = OpenAI(
    api_key=user_secrets.get_secret("Deepseek_api_key"),
    base_url="https://api.deepseek.com"
)

def format_grid(grid):
    return "\n".join([f"R{i}: {row}" for i, row in enumerate(grid)])

def save_dual_outputs(data_dict):
    # Save Raw JSON
    with open(JSON_OUTPUT, 'w') as f:
        json.dump(data_dict, f, indent=2)
        
    # Process into structured CSV
    headers = [
        'Task_ID', 
        'All_Used_Transformations', 
        'Total_Transformations', 
        'Spatial_Count', 
        'Object_Count', 
        'Color_Count', 
        'Pattern_Count',
        'Primary_Category',
        'Grid_Size_Changed',
        'Estimated_Complexity'
    ]
    
    with open(CSV_OUTPUT, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        
        for tid in sorted(data_dict.keys()):
            data = data_dict[tid]
            
            # Extract categories (handling potential missing keys safely)
            spatial = data.get('Spatial_and_Geometric', [])
            objects = data.get('Object_Based', [])
            color = data.get('Color_and_Logical', [])
            pattern = data.get('Pattern_Recognition', [])
            
            # Aggregate
            all_used = spatial + objects + color + pattern
            all_used_str = " | ".join(all_used)
            total_count = len(all_used)
            
            # Extra fields
            primary = data.get('Primary_Category', 'Unknown')
            grid_changed = data.get('Grid_Size_Changed', False)
            complexity = data.get('Estimated_Complexity', 5)
            
            writer.writerow([
                tid, 
                all_used_str, 
                total_count, 
                len(spatial), 
                len(objects), 
                len(color), 
                len(pattern),
                primary,
                grid_changed,
                complexity
            ])

# --- Engine ---
dataset_dir = '/kaggle/input/competitions/neurogolf-2026'
all_files = sorted(glob.glob(os.path.join(dataset_dir, "task*.json")))
explanations = {}

target_files = [f for f in all_files if START_TASK <= int(''.join(filter(str.isdigit, os.path.basename(f)))) <= END_TASK]
print(f"🚀 Processing {len(target_files)} tasks with Primitive Classification Prompting...")

# --- System Prompt Definition ---
# Defining the strict schema we want DeepSeek to return
SYSTEM_PROMPT = """You are a world-class ARC-AGI pattern recognizer. Your task is to analyze grid transformations and classify the exact logical primitives used.
You MUST output a valid JSON object where the key is the Task ID, and the value follows this exact schema:

{
  "TASK_ID_HERE": {
    "Spatial_and_Geometric": [], // Use ONLY: Rotation, Reflection, Translation, Shifting, Tiling, Cropping, Magnification
    "Object_Based": [],          // Use ONLY: Object Detection, Gravity, Collision Detection, Shape Matching, Sorting, Outlier Detection
    "Color_and_Logical": [],     // Use ONLY: Color Swapping, Background Separation, Flood Fill, Bitwise Logic, Color Mapping
    "Pattern_Recognition": [],   // Use ONLY: Symmetry Completion, Filling Regions, Line Extrapolation, Intersection Finding, Pathfinding, In-painting
    "Primary_Category": "string", // Which of the 4 main categories is most dominant?
    "Grid_Size_Changed": boolean, // Did the dimensions of the grid change between input and output? (true/false)
    "Estimated_Complexity": integer // On a scale of 1-10, how complex is the ONNX implementation likely to be?
  }
}
### COMPLEXITY GUIDELINES (Neurogolf Edition):
- 1-2: ELIMINATION/TRIVIAL. Solvable with a single 1x1 or 3x3 convolution (e.g., simple color swap or 1-pixel shift).
- 3-5: MODERATE. Requires multiple layers or basic pooling (e.g., simple symmetries, local object movement).
- 6-8: HARD. Requires global context or multi-step logic (e.g., gravity, sorting objects, complex tiling).
- 9-10: EXTREME. Very expensive in ONNX/MACs (e.g., Pathfinding, complex recursion, Shape Matching across varying sizes).
Do not invent new primitive names. ONLY use the ones listed above. If a category isn't used, leave its array empty. Output ONLY JSON."""

for i in range(0, len(target_files), BATCH_SIZE):
    batch_files = target_files[i : i + BATCH_SIZE]
    
    prompt_content = "Analyze the following tasks and classify their transformations.\n"
    
    for f in batch_files:
        tid = os.path.basename(f).replace('.json','')
        task_data = json.load(open(f))
        prompt_content += f"\n\n### TASK: {tid}\n"
        for idx, pair in enumerate(task_data['train']):
            prompt_content += f"--- Example {idx} ---\nIN:\n{format_grid(pair['input'])}\nOUT:\n{format_grid(pair['output'])}\n"

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt_content}
                ],
                response_format={'type': 'json_object'}
            )
            
            batch_results = json.loads(response.choices[0].message.content)
            explanations.update(batch_results)
            save_dual_outputs(explanations)
            print(f"✅ Classified: {list(batch_results.keys())}")
            break
        except Exception as e:
            print(f"⚠️ Retry: {e}")
            time.sleep(2)

print(f"🎉 Processed and saved to {JSON_OUTPUT} and {CSV_OUTPUT}")

<a id="output-preview"></a>
## 📊 Output Preview

In [ ]:
import pandas as pd
import json

# 1. Display CSV shortly
print("📊 CSV PREVIEW (Primitives Dataset):")
try:
    # Pointing to the new primitives CSV
    df = pd.read_csv('/kaggle/working/arc_primitives.csv')
    display(df.head(10)) 
except Exception as e:
    print(f"CSV not found: {e}")

print("\n" + "="*50 + "\n")

# 2. Display JSON shortly
print("📄 JSON SNIPPET (First 3 entries):")
try:
    # Pointing to the new primitives JSON
    with open('/kaggle/working/arc_primitives.json', 'r') as f:
        data = json.load(f)
        short_data = {k: data[k] for k in list(data.keys())[:3]}
        print(json.dumps(short_data, indent=2))
except Exception as e:
    print(f"JSON not found: {e}")

<a id="logic-visualizer"></a>
## 🎨 Logic Visualizer

In [ ]:
import json, os, numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors

# --- Settings ---
START_VIS, END_VIS = 1, 10
JSON_INPUT = '/kaggle/working/arc_primitives.json'
DATASET_PATH = '/kaggle/input/competitions/neurogolf-2026' 

cmap = colors.ListedColormap(['#000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00', '#AAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'])
norm = colors.Normalize(vmin=0, vmax=9)

with open(JSON_INPUT, 'r') as f:
    primitives_data = json.load(f)

for n in range(START_VIS, END_VIS + 1):
    tid = f"task{n:03d}"
    fpath = os.path.join(DATASET_PATH, f"{tid}.json")
    if not os.path.exists(fpath): continue
        
    with open(fpath, 'r') as f: 
        task_data = json.load(f)
    
    # Extract the structured dictionary for this task
    info = primitives_data.get(tid, {})
    
    if not info:
        print(f"\n⚠️ No primitives found for {tid}. Skipping.")
        continue

    # Flatten the specific primitive arrays into a single list
    all_prims = (
        info.get('Spatial_and_Geometric', []) + 
        info.get('Object_Based', []) + 
        info.get('Color_and_Logical', []) + 
        info.get('Pattern_Recognition', [])
    )
    
    prim_str = " | ".join(all_prims) if all_prims else "None Identified"
    primary = info.get('Primary_Category', 'Unknown')
    complexity = info.get('Estimated_Complexity', '?')
    grid_changed = info.get('Grid_Size_Changed', False)

    # Clean Console Print
    print(f"\n{'='*25} {tid} {'='*25}")
    print(f"PRIMARY CATEGORY : {primary}")
    print(f"PRIMITIVES USED  : {prim_str}")
    print(f"COMPLEXITY       : {complexity}/10")
    print(f"GRID RESIZED     : {grid_changed}\n")

    # Visuals
    train = task_data['train']
    fig, axes = plt.subplots(len(train), 2, figsize=(10, 3 * len(train)))
    if len(train) == 1: axes = [axes]
    
    # Multi-line title containing the metrics
    title_text = f"{tid} | {primary} | Complexity: {complexity}/10\nOps: {prim_str[:100]}{'...' if len(prim_str)>100 else ''}"
    plt.suptitle(title_text, fontsize=11, fontweight='bold')

    for i, pair in enumerate(train):
        for j, key in enumerate(['input', 'output']):
            grid = np.array(pair[key])
            axes[i][j].imshow(grid, cmap=cmap, norm=norm)
            axes[i][j].set_title(f"{key.upper()} {grid.shape}", fontsize=9)
            
            # Add minor ticks and gridlines to show exact pixel boundaries
            axes[i][j].set_xticks(np.arange(-0.5, grid.shape[1], 1), minor=True)
            axes[i][j].set_yticks(np.arange(-0.5, grid.shape[0], 1), minor=True)
            axes[i][j].grid(which='minor', color='white', linestyle='-', linewidth=0.5, alpha=0.5)
            
            # Hide the axis numbers but keep the gridlines
            axes[i][j].tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
            
    plt.tight_layout()
    plt.show()

### 🔗 Dataset Link
* 📊 **[Get the Full Logic Dataset](https://www.kaggle.com/datasets/karnakbaevarthur/neurogolf-2026-task-transformation-library)** – A single file containing all logic types, grid flags, and difficulty levels for 400 tasks.

---

# 🚀 Final Submission

In [ ]:
!pip install onnx==1.21.0 onnxruntime==1.24.4 onnx-tool==1.0.1 numpy==2.4.4

In [ ]:
import io
import os
import sys
import zipfile
from collections import defaultdict
from pathlib import Path

import onnx
from onnx import shape_inference, TensorProto

# ── INPUT ZIPS ────────────────────────────────────────────────────────────────

ZIP_PATHS = [
    "/kaggle/input/notebooks/artemnazemtsev/neurogolf-acking-multiple-tasks-part-3/submission.zip",
    "/kaggle/input/notebooks/vyankteshdwivedi/neurogolf-multi-source-onnx-solver/submission.zip",
]

OUTPUT_ZIP = "/kaggle/working/merged_submission.zip"

# ─────────────────────────────────────────────────────────────────────────────

DISALLOWED_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
MAX_ONNX_BYTES = 1_440_000   # 1.44 MB


def task_num(filename):
    stem = Path(filename).stem.lower()
    if stem.startswith("task") and stem[4:].isdigit():
        return int(stem[4:])
    return None


def fmt_bytes(n):
    for unit in ("B", "KB", "MB"):
        if n < 1024:
            return f"{n:.0f} {unit}"
        n /= 1024
    return f"{n:.1f} MB"


def bar(fraction, width=24):
    filled = round(fraction * width)
    return "█" * filled + "░" * (width - filled)


# ── ONNX VALIDATION ───────────────────────────────────────────────────────────

def validate_onnx(data: bytes) -> tuple[bool, str]:
    """
    Returns (is_valid, reason).
    Checks:
      1. Parses as valid ONNX
      2. File size <= 1.44 MB
      3. No disallowed ops
      4. Shape inference succeeds in strict mode
      5. All tensors have fully static (no dim_param) shapes after inference
      6. Input tensor has all dim_values defined
    """
    if len(data) > MAX_ONNX_BYTES:
        return False, f"file too large ({fmt_bytes(len(data))})"

    try:
        model = onnx.load_from_string(data)
    except Exception as e:
        return False, f"parse error: {e}"

    # check for disallowed ops
    for node in model.graph.node:
        if node.op_type in DISALLOWED_OPS:
            return False, f"disallowed op: {node.op_type}"

    # shape inference (strict)
    try:
        inferred = shape_inference.infer_shapes(model, strict_mode=True)
    except Exception as e:
        return False, f"shape inference failed: {e}"

    # check all value_info + inputs for static shapes (no dim_param)
    all_tensors = (
        list(inferred.graph.input)
        + list(inferred.graph.output)
        + list(inferred.graph.value_info)
    )
    for vi in all_tensors:
        t = vi.type
        if not t.HasField("tensor_type"):
            continue
        shape = t.tensor_type.shape
        if shape is None:
            continue
        for dim in shape.dim:
            if dim.HasField("dim_param") and dim.dim_param:
                return False, f"dynamic shape (dim_param='{dim.dim_param}') in '{vi.name}'"
            if not dim.HasField("dim_value"):
                return False, f"missing dim_value in '{vi.name}'"

    # check input has all dim_values
    for inp in inferred.graph.input:
        t = inp.type
        if not t.HasField("tensor_type"):
            continue
        shape = t.tensor_type.shape
        if shape is None:
            return False, f"input '{inp.name}' has no shape"
        for dim in shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                return False, f"input '{inp.name}' missing positive dim_value"

    return True, "ok"


# ── LOAD & VALIDATE ───────────────────────────────────────────────────────────

def load_zip(path):
    """
    Returns:
        valid_tasks   : {task_num: bytes}   — passed all checks
        invalid_tasks : {task_num: reason}  — failed at least one check
    """
    valid, invalid = {}, {}
    with zipfile.ZipFile(path, "r") as zf:
        for member in zf.namelist():
            n = task_num(Path(member).name)
            if n is None:
                continue
            data = zf.read(member)
            ok, reason = validate_onnx(data)
            if ok:
                valid[n] = data
            else:
                invalid[n] = reason
    return valid, invalid


# ── MERGE ─────────────────────────────────────────────────────────────────────

def merge_submissions(zip_paths):
    candidate_map = defaultdict(list)   # task_num -> [{src, data, size}]
    per_file      = []

    for path in zip_paths:
        name = Path(path).name
        if not os.path.isfile(path):
            print(f"  [WARN] Not found, skipping: {path}")
            per_file.append({"name": name, "valid": {}, "invalid": {}, "error": "not found"})
            continue
        try:
            valid, invalid = load_zip(path)
        except Exception as exc:
            print(f"  [WARN] Could not read {name}: {exc}")
            per_file.append({"name": name, "valid": {}, "invalid": {}, "error": str(exc)})
            continue

        print(f"  {name:<60}  valid={len(valid):>4}  invalid={len(invalid):>4}  "
              f"({fmt_bytes(os.path.getsize(path))})")
        per_file.append({"name": name, "valid": valid, "invalid": invalid, "error": None})

        for num, data in valid.items():
            candidate_map[num].append({"src": name, "data": data, "size": len(data)})

    merged = {}
    for num, candidates in candidate_map.items():
        best = min(candidates, key=lambda c: c["size"])
        merged[num] = {
            "best_src":     best["src"],
            "data":         best["data"],
            "size":         best["size"],
            "n_candidates": len(candidates),
            "all_sizes":    {c["src"]: c["size"] for c in candidates},
        }

    return merged, per_file


# ── REPORT ────────────────────────────────────────────────────────────────────

def report(merged, per_file):
    all_tasks = sorted(merged.keys())
    n_tasks   = len(all_tasks)
    conflicts = [t for t in all_tasks if merged[t]["n_candidates"] > 1]

    # collect all invalid tasks across all files
    all_invalid = defaultdict(dict)   # task_num -> {src: reason}
    for sf in per_file:
        for t, reason in sf["invalid"].items():
            all_invalid[t][sf["name"]] = reason

    print("\n" + "═" * 72)
    print("  NEUROGOLF SOLUTION MERGER — RESULTS")
    print("═" * 72)
    print(f"  Submissions loaded           : {len(per_file)}")
    print(f"  Valid tasks in merged zip    : {n_tasks}  /  400")
    print(f"  Tasks from single source     : {n_tasks - len(conflicts)}")
    print(f"  Tasks with conflicts         : {len(conflicts)}  (tiebreak = smallest file)")
    print(f"  Unique invalid tasks skipped : {len(all_invalid)}")
    print("═" * 72)

    # per-submission table
    print("\n  PER-SUBMISSION BREAKDOWN\n")
    print(f"  {'File':<60} {'Valid':>6}  {'Invalid':>8}  {'Wins':>5}  {'Overlap':>7}  Coverage")
    print("  " + "-" * 100)
    for sf in per_file:
        if sf["error"] and not sf["valid"]:
            print(f"  {sf['name']:<60}  ERROR: {sf['error']}")
            continue
        n_valid   = len(sf["valid"])
        n_invalid = len(sf["invalid"])
        wins      = sum(1 for t in sf["valid"]
                        if merged.get(t, {}).get("best_src") == sf["name"])
        overlap   = n_valid - wins
        pct       = n_valid / 400
        print(f"  {sf['name']:<60} {n_valid:>6}  {n_invalid:>8}  {wins:>5}  {overlap:>7}  "
              f"{bar(pct)} {pct * 100:5.1f}%")

    # coverage distribution
    print("\n  COVERAGE DISTRIBUTION  (how many submissions have each valid task)\n")
    max_cov = max(merged[t]["n_candidates"] for t in all_tasks) if all_tasks else 0
    for c in range(1, max_cov + 1):
        count = sum(1 for t in all_tasks if merged[t]["n_candidates"] == c)
        b = bar(count / n_tasks, 30) if n_tasks else ""
        label = f"{c} submission{'s' if c > 1 else ' '}"
        print(f"  {label:<16} {b}  {count:>4} tasks  ({count / n_tasks * 100:.1f}%)")

    # conflict detail
    if conflicts:
        print(f"\n  CONFLICTS — {len(conflicts)} tasks in multiple submissions\n")
        print(f"  {'Task':<10} {'Winner (smallest)':<60} {'Size':>10}  Losers")
        print("  " + "-" * 100)
        for t in conflicts:
            d = merged[t]
            losers = "  |  ".join(
                f"{s} ({fmt_bytes(sz)})"
                for s, sz in d["all_sizes"].items()
                if s != d["best_src"]
            )
            print(f"  task{t:03d}   {d['best_src']:<60} {fmt_bytes(d['size']):>10}  {losers}")

    # invalid tasks summary
    if all_invalid:
        print(f"\n  SKIPPED TASKS — {len(all_invalid)} tasks failed validation\n")
        # group by reason category
        reason_groups = defaultdict(list)
        for t, srcs in all_invalid.items():
            # use first reason found
            first_reason = next(iter(srcs.values()))
            if "dim_param" in first_reason:
                cat = "dynamic shape (dim_param)"
            elif "missing dim_value" in first_reason:
                cat = "missing dim_value"
            elif "shape inference" in first_reason:
                cat = "shape inference failed"
            elif "parse error" in first_reason:
                cat = "parse error"
            elif "disallowed op" in first_reason:
                cat = "disallowed op"
            elif "file too large" in first_reason:
                cat = "file too large"
            else:
                cat = "other"
            reason_groups[cat].append(t)

        for cat, tasks in sorted(reason_groups.items()):
            tasks_str = ", ".join(str(t) for t in sorted(tasks))
            print(f"  [{cat}]  {len(tasks)} tasks")
            print(f"    tasks: {tasks_str}\n")

    print()


# ── WRITE OUTPUT ZIP ──────────────────────────────────────────────────────────

def write_zip(merged, output_path):
    os.makedirs(Path(output_path).parent, exist_ok=True)
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for num in sorted(merged.keys()):
            zf.writestr(f"task{num:03d}.onnx", merged[num]["data"])
    size = os.path.getsize(output_path)
    print(f"  Wrote {len(merged)} tasks  →  {output_path}  ({fmt_bytes(size)})\n")


# ── RUN ───────────────────────────────────────────────────────────────────────

print(f"\nReading and validating {len(ZIP_PATHS)} submission(s) …\n")
merged, per_file = merge_submissions(ZIP_PATHS)

if not merged:
    print("[ERROR] No valid tasks found. Check your ZIP_PATHS above.")
    sys.exit(1)

report(merged, per_file)
write_zip(merged, OUTPUT_ZIP)